# 01 — Schema Inference Experiments
Validates DataPilot's two-stage column type inference:
1. **TypeInferencer** — rule-based, < 5ms for 50 columns, zero LLM calls
2. **SemanticClassifier** — Claude Haiku batch call for ambiguous columns only

**Real-time integration**: each resolved column fires a `schema:column_classified`
Socket.IO event. This notebook measures classification accuracy and latency
so we know how many columns require LLM disambiguation and what the expected
`schema:complete` event delay will be.


In [1]:
import sys
import subprocess
from pathlib import Path

try:
    import pandas as pd
    import polars as pl
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "polars", "openpyxl", "pyarrow", "-q"])
    import pandas as pd
    import polars as pl

repo_root = None
for candidate in [Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent]:
    if (candidate / "notebooks" / "datasets").exists():
        repo_root = candidate
        break

if repo_root is None:
    repo_root = Path.cwd()

if (repo_root / "notebooks" / "datasets").exists():
    data_dir = repo_root / "notebooks" / "datasets"
else:
    data_dir = repo_root / "datasets"

sys.path.insert(0, str(repo_root.resolve()))

from backend.analytics_engine.ingestion.file_reader import FileReader

reader = FileReader()

retail_path = data_dir / "sample_retail_sales.csv"
hr_path = data_dir / "sample_hr_data.xlsx"
iot_path = data_dir / "sample_iot_timeseries.parquet"

retail = await reader.read_path(str(retail_path))
hr = await reader.read_path(str(hr_path))
iot = await reader.read_path(str(iot_path))

print(f"Retail: {retail.shape}  |  HR: {hr.shape}  |  IoT: {iot.shape}")

2026-07-06 21:59:14 [debug    ] format_detected                delimiter="','" encoding=ascii filename=sample_retail_sales.csv format=csv
2026-07-06 21:59:14 [info     ] csv_read_polars                cols=10 rows=500
2026-07-06 21:59:14 [debug    ] format_detected                delimiter="','" encoding=utf-8 filename=sample_hr_data.xlsx format=excel
2026-07-06 21:59:14 [warning  ] polars_excel_failed_fallback_pandas error="required package 'fastexcel' not found.\nPlease install using the command `pip install fastexcel`."
2026-07-06 21:59:15 [debug    ] format_detected                delimiter="','" encoding=utf-8 filename=sample_iot_timeseries.parquet format=parquet
2026-07-06 21:59:15 [info     ] parquet_read_polars            cols=10 rows=8760
Retail: (500, 10)  |  HR: (300, 11)  |  IoT: (8760, 10)


## Rule-based TypeInferencer — speed and accuracy

In [4]:
import time

from backend.agents.data.schema.type_inferencer import infer_all_columns

results = {}
for name, df in [('retail', retail), ('hr', hr), ('iot', iot)]:
    start = time.perf_counter()
    inferences = infer_all_columns(df)
    elapsed_ms = (time.perf_counter() - start) * 1000

    needs_llm = sum(1 for i in inferences if i.needs_llm)
    results[name] = {
        'cols': len(inferences),
        'needs_llm': needs_llm,
        'pct_rule_only': round((1 - needs_llm/len(inferences)) * 100, 1),
        'elapsed_ms': round(elapsed_ms, 2),
    }

    print(f"\n── {name.upper()} ({len(inferences)} columns, {elapsed_ms:.1f}ms) ──")
    for inf in inferences:
        flag = '🤖 LLM' if inf.needs_llm else '✓ Rule'
        print(f"  {flag}  {inf.column_name:<25} → {inf.semantic_type:<20} (conf={inf.confidence:.2f})")

print("\n── Summary ──")
for name, r in results.items():
    print(f"  {name}: {r['pct_rule_only']}% rule-only, {r['elapsed_ms']}ms total")



── RETAIL (10 columns, 43.6ms) ──
  ✓ Rule  order_id                  → identifier           (conf=0.75)
  ✓ Rule  order_date                → datetime             (conf=0.70)
  ✓ Rule  region                    → categorical          (conf=0.80)
  ✓ Rule  category                  → categorical          (conf=0.80)
  🤖 LLM  product_sku               → free_text            (conf=0.50)
  ✓ Rule  quantity                  → numeric_count        (conf=0.85)
  ✓ Rule  unit_price                → currency             (conf=0.90)
  ✓ Rule  revenue                   → currency             (conf=0.90)
  ✓ Rule  discount_pct              → percentage           (conf=0.85)
  ✓ Rule  status                    → categorical          (conf=0.80)

── HR (11 columns, 13.5ms) ──
  🤖 LLM  employee_id               → free_text            (conf=0.50)
  ✓ Rule  first_name                → categorical          (conf=0.80)
  ✓ Rule  last_name                 → categorical          (conf=0.80)
  🤖 LLM  emai

## Ambiguous column analysis — what the LLM needs to classify

In [5]:
for name, df in [('retail', retail), ('hr', hr), ('iot', iot)]:
    inferences = infer_all_columns(df)
    ambiguous = [i for i in inferences if i.needs_llm]
    if ambiguous:
        print(f"\n── {name.upper()} — {len(ambiguous)} ambiguous columns ──")
        for inf in ambiguous:
            print(f"  {inf.column_name}: dtype={inf.data_type}, "
                  f"samples={inf.sample_values[:3]}, "
                  f"unique={inf.unique_count}, null_rate={inf.null_rate:.1%}")



── RETAIL — 1 ambiguous columns ──
  product_sku: dtype=String, samples=['SKU-9607', 'SKU-4426', 'SKU-8246'], unique=483, null_rate=0.0%

── HR — 2 ambiguous columns ──
  employee_id: dtype=object, samples=['EMP-0001', 'EMP-0002', 'EMP-0003'], unique=300, null_rate=0.0%
  email: dtype=object, samples=['emp0001@company.com', 'emp0002@company.com', 'emp0003@company.com'], unique=300, null_rate=0.0%


## Socket.IO event simulation — per-column classification latency

In [9]:
import asyncio
import time

# Simulate what the SchemaAgent emits to the Socket.IO room
# so we can measure the expected UX: time from file upload → last schema event

class MockSio:
    def __init__(self):
        self.events = []
        self.start  = time.perf_counter()
    async def emit(self, event, data, room=None):
        self.events.append({
            'event': event,
            'col':   data.get('column_name', ''),
            'type':  data.get('semantic_type', ''),
            'ms':    round((time.perf_counter() - self.start) * 1000, 1),
        })

async def simulate_schema_events(df, name):
    from backend.agents.data.schema.type_inferencer import infer_all_columns
    sio = MockSio()
    inferences = infer_all_columns(df)
    for inf in inferences:
        await sio.emit('schema:column_classified', {
            'column_name':   inf.column_name,
            'semantic_type': inf.semantic_type,
            'confidence':    inf.confidence,
        }, room=f'dataset:test_{name}')
    await sio.emit('schema:complete', {'column_count': len(inferences)})
    return sio.events

for name, df in [('retail', retail), ('hr', hr)]:
    events = await simulate_schema_events(df, name)
    last_col = next(e for e in reversed(events) if e['event'] == 'schema:column_classified')
    complete = next(e for e in events if e['event'] == 'schema:complete')
    print(f"\n── {name.upper()} ──")
    print(f"  First column classified at: {events[0]['ms']}ms")
    print(f"  Last column classified at:  {last_col['ms']}ms")
    print(f"  schema:complete at:         {complete['ms']}ms")
    print(f"  → Browser sees first card in {events[0]['ms']}ms "
          f"(not {complete['ms']}ms — real-time advantage!)")



── RETAIL ──
  First column classified at: 2.0ms
  Last column classified at:  2.0ms
  schema:complete at:         2.0ms
  → Browser sees first card in 2.0ms (not 2.0ms — real-time advantage!)

── HR ──
  First column classified at: 10.1ms
  Last column classified at:  10.1ms
  schema:complete at:         10.1ms
  → Browser sees first card in 10.1ms (not 10.1ms — real-time advantage!)


## Cardinality threshold sensitivity — finding the best categorical cutoff

In [10]:
# Test different cardinality_ratio thresholds for categorical detection
# The right threshold minimises misclassification of free_text as categorical

test_thresholds = [0.02, 0.05, 0.08, 0.10, 0.15]
df = retail

for threshold in test_thresholds:
    cats = [
        col for col in df.columns
        if df[col].dtype == pl.Utf8
        and df[col].n_unique() / max(len(df) - df[col].null_count(), 1) <= threshold
    ]
    print(f"  threshold={threshold:.2f}: {len(cats)} categorical columns: {cats}")


  threshold=0.02: 3 categorical columns: ['region', 'category', 'status']
  threshold=0.05: 3 categorical columns: ['region', 'category', 'status']
  threshold=0.08: 3 categorical columns: ['region', 'category', 'status']
  threshold=0.10: 3 categorical columns: ['region', 'category', 'status']
  threshold=0.15: 3 categorical columns: ['region', 'category', 'status']
